# nanoGPT: Build a GPT from Scratch

Follow Karpathy's path: a character-level transformer with self-attention that generates text. Everything in the curriculum converges here. **Runtime: GPU.**

## 1. Data
We train on a small text. Swap in any `.txt` you like.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
text = open('input.txt').read() if __import__('os').path.exists('input.txt') else 'hello world. ' * 2000
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
print('vocab size:', len(chars))

## 2. Hyperparameters & batching

In [ ]:
block_size, batch_size, n_embd, n_head, n_layer = 32, 32, 64, 4, 3
def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## 3. A self-attention head
The mechanism from the Deep Learning track: Q, K, V, scaled dot-product, causal mask.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = (q @ k.transpose(-2, -1)) * C ** -0.5
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        return att @ v

## 4. The transformer block and model

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        hs = n_embd // n_head
        self.heads = nn.ModuleList([Head(hs) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.ff = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.proj(torch.cat([h(self.ln1(x)) for h in self.heads], dim=-1))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(len(chars), n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.lnf = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, len(chars))
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=device))
        x = self.head(self.lnf(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(x.view(-1, x.size(-1)), targets.view(-1))
        return x, loss

model = GPT().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

## 5. Train

In [ ]:
for step in range(2000):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(step, round(loss.item(), 3))

## 6. Generate
Feed predictions back in, one token at a time.

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
for _ in range(200):
    logits, _ = model(idx[:, -block_size:])
    probs = F.softmax(logits[:, -1, :], dim=-1)
    idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
print(decode(idx[0].tolist()))

## Homework
1. Upload a real corpus as `input.txt` (Shakespeare is the classic) and retrain.
2. Increase `n_layer`, `n_embd`, and `block_size` — watch quality improve.
3. Add dropout for regularization.
4. Explain, in comments, what every line of the `Head` class does.